In [34]:
import yfinance as yf
import pandas as pd
import talib
import torch

In [2]:
# data = yf.download("^GDAXI", start="2019-01-01", end="2024-01-01")
# data.to_csv('index_stock.csv')

df = pd.read_csv('index_stock.csv')
print(df.shape)
print(df.head())


(1274, 6)
        Price             Close              High               Low  \
0      Ticker            ^GDAXI            ^GDAXI            ^GDAXI   
1        Date               NaN               NaN               NaN   
2  2019-01-02  10580.1904296875  10612.7197265625  10386.9697265625   
3  2019-01-03    10416.66015625    10538.66015625  10400.1103515625   
4  2019-01-04  10767.6904296875    10786.33984375   10483.900390625   

               Open    Volume  
0            ^GDAXI    ^GDAXI  
1               NaN       NaN  
2    10477.76953125  79626700  
3  10467.1103515625  84733800  
4  10533.9404296875  95339500  


In [3]:
df = df.iloc[2:]

In [4]:
df

,Price,Close,High,Low,Open,Volume
2,2019-01-02,10580.1904296875,10612.7197265625,10386.9697265625,10477.76953125,79626700
3,2019-01-03,10416.66015625,10538.66015625,10400.1103515625,10467.1103515625,84733800
4,2019-01-04,10767.6904296875,10786.33984375,10483.900390625,10533.9404296875,95339500
5,2019-01-07,10747.8095703125,10814.4697265625,10681.26953125,10814.3896484375,71151400
6,2019-01-08,10803.98046875,10910.7099609375,10745.0302734375,10750.1904296875,93672200
...,...,...,...,...,...,...
1269,2023-12-21,16687.419921875,16708.349609375,16624.16015625,16667.310546875,57871300
1270,2023-12-22,16706.1796875,16735.3203125,16651.779296875,16673.30078125,46295300
1271,2023-12-27,16742.0703125,16775.7109375,16697.580078125,16727.76953125,37678900
1272,2023-12-28,16701.55078125,16783.7890625,16688.51953125,16780.94921875,36091600


In [5]:
df['Doji'] = talib.CDLDOJI(df['Open'], df['High'], df['Low'], df['Close'])
df['Hammer'] = talib.CDLHAMMER(df['Open'], df['High'], df['Low'], df['Close'])
df['Engulfing'] = talib.CDLENGULFING(df['Open'], df['High'], df['Low'], df['Close'])


In [6]:
def min_max_normalization(x, columns=[]):
    x = x.astype(float)
    x_scaled = (x - x.min()) / (x.max() - x.min())
    return x_scaled


In [7]:
df[['Close', 'High', 'Low', 'Open', 'Volume']] = df[['Close', 'High', 'Low', 'Open', 'Volume']].apply(min_max_normalization)

In [8]:
df = df.rename(columns = {'Price':'Date'})

In [9]:
df = df.reset_index()

In [11]:
df.drop(columns='index')
df.head(20)

,index,Date,Close,High,Low,Open,Volume,Doji,Hammer,Engulfing
0,2,2019-01-02,0.256022,0.233268,0.250612,0.234457,0.202463,0,0,0
1,3,2019-01-03,0.236444,0.224382,0.252157,0.233196,0.215566,0,0,0
2,4,2019-01-04,0.278470,0.254098,0.262009,0.241102,0.242776,0,0,0
3,5,2019-01-07,0.276090,0.257473,0.285217,0.274280,0.180718,0,0,0
4,6,2019-01-08,0.282815,0.269020,0.292714,0.266685,0.238499,0,0,0
5,7,2019-01-09,0.293510,0.275164,0.303774,0.282604,0.244656,0,0,0
6,8,2019-01-10,0.296895,0.270990,0.297773,0.277770,0.196379,0,0,0
7,9,2019-01-11,0.292809,0.274814,0.302696,0.290220,0.191970,0,0,0
8,10,2019-01-14,0.289032,0.266083,0.297620,0.275044,0.167362,0,0,0
9,11,2019-01-15,0.293327,0.279113,0.300658,0.291855,0.203927,0,0,0


In [14]:
df.value_counts(['Doji', 'Hammer', 'Engulfing'])

Doji  Hammer  Engulfing
0     0        0           997
100   0        0           177
0     0       -100          45
               100          28
      100      0            22
100   100      0             3
Name: count, dtype: int64

In [20]:
def define_pattern(x):
    if x['Doji'] == 0 and x['Hammer'] == 0 and x['Engulfing'] == 0:
        return 0
    elif x['Doji'] == 100 and x['Hammer'] == 0 and x['Engulfing'] == 0:
        return 1
    elif x['Doji'] == 0 and x['Hammer'] == 100 and x['Engulfing'] == 0:
        return 2
    elif x['Doji'] == 0 and x['Hammer'] == 0 and (x['Engulfing'] != 0):
        return 3
    else:
        return -1 
    


In [21]:
df['Pattern'] = df.apply(define_pattern, axis=1)

In [22]:
df

,index,Date,Close,High,Low,Open,Volume,Doji,Hammer,Engulfing,Pattern
0,2,2019-01-02,0.256022,0.233268,0.250612,0.234457,0.202463,0,0,0,0
1,3,2019-01-03,0.236444,0.224382,0.252157,0.233196,0.215566,0,0,0,0
2,4,2019-01-04,0.278470,0.254098,0.262009,0.241102,0.242776,0,0,0,0
3,5,2019-01-07,0.276090,0.257473,0.285217,0.274280,0.180718,0,0,0,0
4,6,2019-01-08,0.282815,0.269020,0.292714,0.266685,0.238499,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
1267,1269,2023-12-21,0.987189,0.964615,0.984013,0.966700,0.146646,0,0,0,0
1268,1270,2023-12-22,0.989435,0.967851,0.987261,0.967409,0.116946,0,0,0,0
1269,1271,2023-12-27,0.993731,0.972697,0.992646,0.973853,0.094839,0,0,0,0
1270,1272,2023-12-28,0.988880,0.973666,0.991581,0.980144,0.090766,0,0,-100,3


In [25]:
df['Pattern'].value_counts()

Pattern
 0    997
 1    177
 3     73
 2     22
-1      3
Name: count, dtype: int64

In [31]:
df = df[df['Pattern'] != -1]
df = df.drop(columns =['Doji', 'Hammer', 'Engulfing'])

In [32]:
df.head(20)

,index,Date,Close,High,Low,Open,Volume,Pattern
0,2,2019-01-02,0.256022,0.233268,0.250612,0.234457,0.202463,0
1,3,2019-01-03,0.236444,0.224382,0.252157,0.233196,0.215566,0
2,4,2019-01-04,0.278470,0.254098,0.262009,0.241102,0.242776,0
3,5,2019-01-07,0.276090,0.257473,0.285217,0.274280,0.180718,0
4,6,2019-01-08,0.282815,0.269020,0.292714,0.266685,0.238499,0
5,7,2019-01-09,0.293510,0.275164,0.303774,0.282604,0.244656,0
6,8,2019-01-10,0.296895,0.270990,0.297773,0.277770,0.196379,0
7,9,2019-01-11,0.292809,0.274814,0.302696,0.290220,0.191970,0
8,10,2019-01-14,0.289032,0.266083,0.297620,0.275044,0.167362,0
9,11,2019-01-15,0.293327,0.279113,0.300658,0.291855,0.203927,0


In [35]:
W1 = torch.randn(5,8,requires_grad = True)
b1 = torch.zeros(8,requires_grad = True)
W2 = torch.randn(8,4,requires_grad = True)
b2 = torch.zeros(4,requires_grad = True)

In [36]:
def forward_pass(X):
    hidden_lay_pre_act = torch.matmul(X, W1) + b1
    relu_act = torch.relu(hidden_lay_pre_act)
    output_lay_pre_act = torch.matmul(relu_act, W2) + b2
    softmax_act = torch.softmax(output_lay_pre_act, dim=1)
    return softmax_act